# Étape 4a — Balayages sans entraînement

**Projet 5 · LoRA Zellige** — Google Colab, GPU T4.

Deux des trois balayages de l'Étape 4 ne coûtent **aucun entraînement** :

| Balayage | Pourquoi c'est gratuit |
|---|---|
| **Nombre de pas** 250 / 500 / 750 / 1000 | les points de contrôle de l'Étape 3 existent déjà |
| **Intensité** 0,25 → 1,25 | l'intensité se règle à la génération, pas à l'entraînement |

C'est le bénéfice d'avoir sauvegardé les checkpoints et rendu l'intensité
réglable. Le balayage de rang, lui, demande 4 entraînements : c'est le
notebook 05.

Durée : environ 50 minutes.

> **Exécution → Modifier le type d'exécution → GPU T4**


## 1. Environnement


In [ ]:
# torchao 0.10 est preinstalle sur Colab et peft exige >= 0.16 :
# sans cette desinstallation, le branchement de la LoRA echoue.
!pip -q uninstall -y torchao 2>&1 | tail -1

!pip -q install 'diffusers>=0.27' 'peft>=0.10' 'transformers>=4.38' \
    accelerate safetensors bitsandbytes lpips pyyaml 2>&1 | tail -2

import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())


## 2. Dépôt et jeu de données


In [ ]:
!git clone -q https://github.com/HatimTj/lora-zellige-patrimoine.git /content/projet 2>/dev/null || (cd /content/projet && git fetch -q && git reset -q --hard origin/main)
%cd /content/projet
!python src/collect_wikimedia.py --reconstruire \
  && python src/prepare_data.py \
  && python src/legender.py


In [ ]:
import subprocess

def lancer(commande):
    """Execute une commande et s'arrete net si elle echoue."""
    print('>>', commande, flush=True)
    code = subprocess.run(commande, shell=True).returncode
    assert code == 0, f'echec (code {code}) : {commande}'


## 3. Récupérer la LoRA de l'Étape 3

Les poids ne sont pas dans Git — le sujet interdit d'y mettre des
modèles. Il faut donc les apporter dans la session.

**Avant de lancer « Tout exécuter » :**

1. ouvrez le panneau **Fichiers** (icône dossier, barre de gauche)
2. glissez-y **`runs_r8.zip`** depuis votre Mac (22,9 Mo, à la racine
   du projet)
3. attendez la fin du téléversement (barre de progression en bas)

Le fichier se dépose dans `/content/`. La cellule suivante le trouve
toute seule, où qu'il soit, et le décompresse au bon endroit.

> Si vous oubliez cette étape, la cellule vous proposera une boîte de
> dialogue — mais elle interrompt « Tout exécuter », donc il faudra
> reprendre à la main ensuite.


In [ ]:
from pathlib import Path
import shutil, zipfile

FINAL = Path('runs/r8/pytorch_lora_weights.safetensors')
Path('runs/r8').mkdir(parents=True, exist_ok=True)

def installer(archive):
    """Decompresse une archive de poids a la racine du projet."""
    zipfile.ZipFile(archive).extractall('.')
    print('decompresse :', archive)

if FINAL.exists():
    print('poids deja presents')
else:
    # On cherche l'archive la ou Colab depose les fichiers glisses
    trouvees = sorted(Path('/content').glob('*.zip')) + \
               sorted(Path('.').glob('*.zip'))
    if trouvees:
        installer(trouvees[0])
    else:
        # Repli interactif : interrompt « Tout executer »
        print('aucune archive trouvee, ouverture du televersement...')
        from google.colab import files
        for nom in files.upload():
            if nom.endswith('.zip'):
                installer(nom)
            elif nom.endswith('.safetensors'):
                shutil.move(nom, FINAL)

for f in sorted(Path('runs/r8').glob('**/*.safetensors')):
    print(f'  {f}  ({f.stat().st_size/1e6:.1f} Mo)')


In [ ]:
from pathlib import Path
attendus = ['runs/r8/pytorch_lora_weights.safetensors',
            'runs/r8/checkpoint-250/pytorch_lora_weights.safetensors',
            'runs/r8/checkpoint-500/pytorch_lora_weights.safetensors',
            'runs/r8/checkpoint-750/pytorch_lora_weights.safetensors']
manquants = [c for c in attendus if not Path(c).exists()]
assert not manquants, f'fichiers manquants : {manquants}'
print('les 4 jeux de poids sont la')


## 4. Balayage du nombre de pas

Même rang (8), même intensité (1,0), seule la durée d'entraînement change.

La question : **à partir de quand entraîner plus n'apporte plus rien — et
commence à faire mémoriser ?**

4 configurations × 40 images ≈ 18 minutes.


In [ ]:
for pas, dossier in [(250, 'runs/r8/checkpoint-250'),
                     (500, 'runs/r8/checkpoint-500'),
                     (750, 'runs/r8/checkpoint-750'),
                     (1000, 'runs/r8')]:
    nom = f'r8_{pas}pas'
    lancer(f'python src/generate.py --lora {dossier} --intensite 1.0 --sortie results/{nom}')
    lancer(f'python src/evaluate.py --dossier results/{nom} --nom {nom} --rang 8 --pas {pas}')


## 5. Balayage de l'intensité

Même LoRA (rang 8, 1000 pas), seul le curseur change.

C'est le balayage le plus parlant en soutenance : **une seule LoRA, cinq
comportements**. À 0,25 le modèle obéit au texte mais ignore le style ; à
1,25 il impose son style et oublie ce qu'on lui demande.

L'hypothèse à vérifier : la tasse à café (T1), ratée à 1,0, devrait
réapparaître vers 0,5.

5 configurations ≈ 23 minutes.


In [ ]:
for intensite in [0.25, 0.50, 0.75, 1.00, 1.25]:
    nom = f'intensite_{intensite:.2f}'
    lancer(f'python src/generate.py --lora runs/r8 --intensite {intensite} --sortie results/{nom}')
    lancer(f'python src/evaluate.py --dossier results/{nom} --nom {nom} --rang 8 --pas 1000')


## 6. Les figures


In [ ]:
!python src/figures.py

from IPython.display import Image, display
for f in ['pas.png', 'intensite.png', 'compromis.png']:
    print('—' * 70); print(f)
    display(Image(f'results/figures/{f}'))


## 7. La planche d'intensité

Le même prompt à cinq intensités, pour voir le compromis au lieu de le lire.


In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

intensites = [0.25, 0.50, 0.75, 1.00, 1.25]
lignes = ['D1', 'T1', 'H1']        # domaine, transfert, hors domaine
fig, axes = plt.subplots(len(lignes), len(intensites),
                         figsize=(2.7 * len(intensites), 2.9 * len(lignes)))
for i, pid in enumerate(lignes):
    for j, inten in enumerate(intensites):
        ax = axes[i, j]
        ax.imshow(PILImage.open(f'results/intensite_{inten:.2f}/{pid}_s1234.png'))
        ax.axis('off')
        if i == 0: ax.set_title(f'intensite {inten:.2f}', fontsize=10)
        if j == 0: ax.text(-0.08, 0.5, pid, transform=ax.transAxes,
                           rotation=90, va='center', fontsize=11, weight='bold')
plt.suptitle('Une seule LoRA, cinq intensites - graine 1234', fontsize=13)
plt.tight_layout(); plt.savefig('results/figures/planche_intensite.png', dpi=110)
plt.show()


## 8. Le tableau complet


In [ ]:
import pandas as pd
pd.set_option('display.width', 200)
df = pd.read_csv('results/results.csv')
df[['nom', 'rang', 'pas', 'intensite', 'style_dino_moy', 'prompt_domaine',
    'prompt_transfert', 'prompt_hors_domaine', 'memorisation_max_dino',
    'diversite_lpips']]


## 9. Sauvegarder


In [ ]:
from google.colab import userdata
jeton = userdata.get('GITHUB_TOKEN')
url = f'https://{jeton}@github.com/HatimTj/lora-zellige-patrimoine.git'

!git config user.name 'HatimTj'
!git config user.email 'hatimtajimi1@gmail.com'
!git add results/results.csv results/figures/*.png
!git add results/*/manifeste.json results/*/evaluation.json
!git commit -m 'etape4: balayages du nombre de pas et de l intensite'
!git pull --rebase -q {url} main
!git push {url} main


> Puis **clic droit → Effacer la sortie** sur la cellule ci-dessus.
